# Imbalanced classes experiment


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import random
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

# use one seed so that the same cat images are removed every time.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


PROJECT_PATH = os.environ.get("PROJECT_PATH", "/content/drive/MyDrive/deep learning/DD2424_Project1")
DATA_PATH = os.environ.get("DATA_PATH", "/content/data")
os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
Using device: cuda
GPU: Tesla T4


## 1. Load the dataset



In [ ]:
weights = models.ResNet18_Weights.DEFAULT
imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

plain_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# train_transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
# ])

train_breed_set = datasets.OxfordIIITPet(
    root=DATA_PATH,
    split="trainval",
    target_types="category",
    download=True,
    transform=plain_transform,
)

test_breed_set = datasets.OxfordIIITPet(
    root=DATA_PATH,
    split="test",
    target_types="category",
    download=True,
    transform=plain_transform,
)

test_loader = DataLoader(
    test_breed_set,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

class_names = train_breed_set.classes
num_classes = len(class_names)

print("Train images:", len(train_breed_set))
print("Test images:", len(test_breed_set))
print("Number of classes:", num_classes)


100%|██████████| 792M/792M [00:36<00:00, 21.8MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 12.5MB/s]


Train images: 3680
Test images: 3669
Number of classes: 37


## 2. Construct the imbalanced training set

 keeping **100% of dog-breed training images** and **20% of cat-breed training images**.


In [ ]:
cat_breeds = [
    "Abyssinian",
    "Bengal",
    "Birman",
    "Bombay",
    "British Shorthair",
    "Egyptian Mau",
    "Maine Coon",
    "Persian",
    "Ragdoll",
    "Russian Blue",
    "Siamese",
    "Sphynx",
]

# lists of class ids for cats and dogs
cat_class_ids = []
dog_class_ids = []

for class_id in range(len(class_names)):
    class_name = class_names[class_id]
    if class_name in cat_breeds:
        cat_class_ids.append(class_id)
    else:
        dog_class_ids.append(class_id)

print("Cat classes:")
for class_id in cat_class_ids:
    print(class_id, class_names[class_id])

print("Number of cat classes:", len(cat_class_ids))
print("Number of dog classes:", len(dog_class_ids))

train_labels = []
for idx in range(len(train_breed_set)):
    image, label = train_breed_set[idx]
    train_labels.append(label)

class_to_indices = defaultdict(list)
for idx in range(len(train_labels)):
    label = train_labels[idx]
    class_to_indices[label].append(idx)


# - keep all dogs
# - keep only 5% of each cat breed
imbalanced_indices = []
cat_fraction = 0.05

for class_id in class_to_indices:
    indices_for_this_class = class_to_indices[class_id]

    if class_id in cat_class_ids:
        number_to_keep = int(len(indices_for_this_class) * cat_fraction)
        number_to_keep = max(1, number_to_keep)
        selected_indices = random.sample(indices_for_this_class, number_to_keep)
    else:
        selected_indices = list(indices_for_this_class)

    for idx in selected_indices:
        imbalanced_indices.append(idx)

random.shuffle(imbalanced_indices)

imbalanced_train_set = Subset(train_breed_set, imbalanced_indices)

imbalanced_labels = []
for idx in imbalanced_indices:
    imbalanced_labels.append(train_labels[idx])

imbalanced_train_loader = DataLoader(
    imbalanced_train_set,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

print("Original training size:", len(train_breed_set))
print("Imbalanced training size:", len(imbalanced_train_set))


Cat classes:
0 Abyssinian
5 Bengal
6 Birman
7 Bombay
9 British Shorthair
11 Egyptian Mau
20 Maine Coon
23 Persian
26 Ragdoll
27 Russian Blue
32 Siamese
33 Sphynx
Number of cat classes: 12
Number of dog classes: 25
Original training size: 3680
Imbalanced training size: 2549


In [ ]:
original_counts = Counter(train_labels)
imbalanced_counts = Counter(imbalanced_labels)

rows = []
for class_id in range(num_classes):
    if class_id in cat_class_ids:
        animal_type = "cat"
    else:
        animal_type = "dog"

    row = {
        "class_id": class_id,
        "class_name": class_names[class_id],
        "animal_type": animal_type,
        "original_train_count": original_counts[class_id],
        "imbalanced_train_count": imbalanced_counts[class_id],
    }
    rows.append(row)

dist_df = pd.DataFrame(rows)
display(dist_df)


,class_id,class_name,animal_type,original_train_count,imbalanced_train_count
0,0,Abyssinian,cat,100,5
1,1,American Bulldog,dog,100,100
2,2,American Pit Bull Terrier,dog,100,100
3,3,Basset Hound,dog,100,100
4,4,Beagle,dog,100,100
5,5,Bengal,cat,100,5
6,6,Birman,cat,100,5
7,7,Bombay,cat,96,4
8,8,Boxer,dog,100,100
9,9,British Shorthair,cat,100,5


## 3. Training and evaluation functions



In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)


        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        # loss=
        loss.backward()
        optimizer.step()

        running_loss = running_loss + loss.item()

        predicted = torch.argmax(outputs, dim=1)
        total = total + labels.size(0)
        correct = correct + (predicted == labels).sum().item()

    average_loss = running_loss / len(loader)
    accuracy = 100.0 * correct / total
    # print("Accuracy: ", accuracy)
    # print("At epoch: ")
    return average_loss, accuracy


def evaluate_detailed(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    all_predictions = []
    all_true_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss = running_loss + loss.item()

            predicted = torch.argmax(outputs, dim=1)



            for value in predicted.cpu().numpy():
                all_predictions.append(value)
            for value in labels.cpu().numpy():
                all_true_labels.append(value)

    all_predictions = np.array(all_predictions)
    all_true_labels = np.array(all_true_labels)
    correct_total = (all_predictions == all_true_labels).sum()
    overall_accuracy = 100.0 * correct_total / len(all_true_labels)


    #
    per_class_rows = []
    f1_values = []
    support_values = []

    for class_id in range(num_classes):
        class_name = class_names[class_id]

        true_mask = all_true_labels == class_id
        pred_mask = all_predictions == class_id

        tp = np.logical_and(true_mask, pred_mask).sum()
        fp = np.logical_and(np.logical_not(true_mask), pred_mask).sum()
        fn = np.logical_and(true_mask, np.logical_not(pred_mask)).sum()
        support = true_mask.sum()

        if support > 0:
            class_accuracy = 100.0 * tp / support
        else:
            class_accuracy = 0.0

        if (tp + fp) > 0:
            precision = tp / (tp + fp)
        else:
            precision = 0.0

        if (tp + fn) > 0:
            recall = tp / (tp + fn)
        else:
            recall = 0.0

        if (precision + recall) > 0:
            f1 = 2.0 * precision * recall / (precision + recall)
        else:
            f1 = 0.0

        f1_values.append(f1)
        support_values.append(support)

        per_class_rows.append({
            "class_id": class_id,
            "class_name": class_name,
            "support": support,
            "per_class_accuracy": class_accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    f1_values = np.array(f1_values)
    support_values = np.array(support_values)

    macro_f1 = f1_values.mean()
    weighted_f1 = (f1_values * support_values).sum() / support_values.sum()

    summary = {
        "test_loss": running_loss / len(loader),
        "test_acc": overall_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

    per_class_df = pd.DataFrame(per_class_rows)
    return summary, per_class_df


## 4. Model setup



In [ ]:
def make_model():
    model = models.resnet18(weights=weights)


    number_of_features = model.fc.in_features
    model.fc = nn.Linear(number_of_features, num_classes)

    # freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # unfreeze layer4 and the fc

    for param in model.layer4.parameters():
        param.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    model = model.to(device)
    return model


def run_experiment(experiment_name, train_loader, criterion, num_epochs=8, lr=1e-4):
    model = make_model()

    model = model.to(device)

    # filter(lambda: )
    params_to_update = []
    for param in model.parameters():
        if param.requires_grad:
            params_to_update.append(param)

    optimizer = optim.Adam(params_to_update, lr=lr)

    epoch_results = []
    best_macro_f1 = -1.0
    best_per_class_df = None

    for epoch in range(num_epochs):
        start_time = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        test_summary, per_class_df = evaluate_detailed(model, test_loader, criterion)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        seconds = time.time() - start_time

        result_row = {
            "experiment": experiment_name,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_summary["test_loss"],
            "test_acc": test_summary["test_acc"],
            "macro_f1": test_summary["macro_f1"],
            "weighted_f1": test_summary["weighted_f1"],
            "time_sec": seconds,
        }
        epoch_results.append(result_row)

        if test_summary["macro_f1"] > best_macro_f1:
            best_macro_f1 = test_summary["macro_f1"]
            best_per_class_df = per_class_df.copy()

        print("[", experiment_name, "] Epoch", epoch + 1, "/", num_epochs)
        print("  Train Acc:", round(train_acc, 2), "%")
        print("  Test Acc:", round(test_summary["test_acc"], 2), "%")
        print("  Macro F1:", round(test_summary["macro_f1"], 4))
        print("  Weighted F1:", round(test_summary["weighted_f1"], 4))
        print("  Time:", round(seconds, 1), "seconds")


    best_row = epoch_results[0]
    for row in epoch_results:
        if row["macro_f1"] > best_row["macro_f1"]:
            best_row = row

    print("\nBest result for", experiment_name)
    print("  Test accuracy:", round(best_row["test_acc"], 2), "%")
    print("  Macro F1:", round(best_row["macro_f1"], 4))

    return epoch_results, best_per_class_df


## 5. Run the three experiments



In [ ]:
imbalance_results = []
per_class_results = {}


# 1. baseline on imbalanced training set

criterion_baseline = nn.CrossEntropyLoss()

baseline_results, baseline_per_class_df = run_experiment(
    experiment_name="imbalanced baseline",
    train_loader=imbalanced_train_loader,
    criterion=criterion_baseline,
    num_epochs=8,
    lr=1e-4,
)

for row in baseline_results:
    imbalance_results.append(row)

per_class_results["imbalanced baseline"] = baseline_per_class_df


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


[ imbalanced baseline ] Epoch 1 / 8
  Train Acc: 53.47 %
  Test Acc: 57.02 %
  Macro F1: 0.4908
  Weighted F1: 0.4928
  Time: 35.5 seconds
[ imbalanced baseline ] Epoch 2 / 8
  Train Acc: 89.21 %
  Test Acc: 60.29 %
  Macro F1: 0.5171
  Weighted F1: 0.5192
  Time: 35.1 seconds
[ imbalanced baseline ] Epoch 3 / 8
  Train Acc: 94.9 %
  Test Acc: 60.78 %
  Macro F1: 0.5183
  Weighted F1: 0.5205
  Time: 31.4 seconds
[ imbalanced baseline ] Epoch 4 / 8
  Train Acc: 97.21 %
  Test Acc: 61.73 %
  Macro F1: 0.5315
  Weighted F1: 0.5337
  Time: 31.3 seconds
[ imbalanced baseline ] Epoch 5 / 8
  Train Acc: 99.02 %
  Test Acc: 64.19 %
  Macro F1: 0.5735
  Weighted F1: 0.5759
  Time: 31.5 seconds
[ imbalanced baseline ] Epoch 6 / 8
  Train Acc: 99.88 %
  Test Acc: 68.11 %
  Macro F1: 0.6305
  Weighted F1: 0.6328
  Time: 31.5 seconds
[ imbalanced baseline ] Epoch 7 / 8
  Train Acc: 100.0 %
  Test Acc: 70.67 %
  Macro F1: 0.6739
  Weighted F1: 0.6759
  Time: 31.3 seconds
[ imbalanced baseline ] Epoc

In [ ]:

# 2 weighted cross-entropy


class_counts = np.zeros(num_classes)
for label in imbalanced_labels:
    class_counts[label] = class_counts[label] + 1

# class fewer images - -  larger weight
class_weights = np.zeros(num_classes)
for class_id in range(num_classes):
    class_weights[class_id] = 1.0 / class_counts[class_id]



mean_weight = class_weights.mean()
for class_id in range(num_classes):
    class_weights[class_id] = class_weights[class_id] / mean_weight

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

weight_rows = []
for class_id in range(num_classes):
    if class_id in cat_class_ids:
        animal_type = "cat"
    else:
        animal_type = "dog"

    weight_rows.append({
        "class_id": class_id,
        "class_name": class_names[class_id],
        "animal_type": animal_type,
        "imbalanced_train_count": int(class_counts[class_id]),
        "loss_weight": class_weights[class_id],
    })

weights_df = pd.DataFrame(weight_rows)
display(weights_df)

# weights_csv_path = os.path.join(PROJECT_PATH, "project1_imbalanced_loss_weights.csv")
# weights_df.to_csv(weights_csv_path, index=False)
# print("Saved:", weights_csv_path)

criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_tensor)

weighted_results, weighted_per_class_df = run_experiment(
    experiment_name="imbalanced weighted cross entropy",
    train_loader=imbalanced_train_loader,
    criterion=criterion_weighted,
    num_epochs=8,
    lr=1e-4,
)

for row in weighted_results:
    imbalance_results.append(row)

per_class_results["imbalanced weighted cross entropy"] = weighted_per_class_df


,class_id,class_name,animal_type,imbalanced_train_count,loss_weight
0,0,Abyssinian,cat,5,2.642071
1,1,American Bulldog,dog,100,0.132104
2,2,American Pit Bull Terrier,dog,100,0.132104
3,3,Basset Hound,dog,100,0.132104
4,4,Beagle,dog,100,0.132104
5,5,Bengal,cat,5,2.642071
6,6,Birman,cat,5,2.642071
7,7,Bombay,cat,4,3.302589
8,8,Boxer,dog,100,0.132104
9,9,British Shorthair,cat,5,2.642071


Saved: /content/drive/MyDrive/deep learning/DD2424_Project1/project1_imbalanced_loss_weights.csv
[ imbalanced weighted cross entropy ] Epoch 1 / 8
  Train Acc: 45.59 %
  Test Acc: 66.53 %
  Macro F1: 0.6384
  Weighted F1: 0.6394
  Time: 31.6 seconds
[ imbalanced weighted cross entropy ] Epoch 2 / 8
  Train Acc: 86.9 %
  Test Acc: 74.19 %
  Macro F1: 0.7356
  Weighted F1: 0.7369
  Time: 32.6 seconds
[ imbalanced weighted cross entropy ] Epoch 3 / 8
  Train Acc: 92.55 %
  Test Acc: 74.6 %
  Macro F1: 0.734
  Weighted F1: 0.7352
  Time: 31.5 seconds
[ imbalanced weighted cross entropy ] Epoch 4 / 8
  Train Acc: 95.65 %
  Test Acc: 75.17 %
  Macro F1: 0.7393
  Weighted F1: 0.7408
  Time: 31.1 seconds
[ imbalanced weighted cross entropy ] Epoch 5 / 8
  Train Acc: 97.8 %
  Test Acc: 75.58 %
  Macro F1: 0.7414
  Weighted F1: 0.7426
  Time: 31.4 seconds
[ imbalanced weighted cross entropy ] Epoch 6 / 8
  Train Acc: 98.82 %
  Test Acc: 74.84 %
  Macro F1: 0.7335
  Weighted F1: 0.7349
  Time: 31

In [ ]:

# 3 oversampling


# smaller class sampled more often
sample_weight_list = []
for label in imbalanced_labels:
    this_weight = 1.0 / class_counts[label]
    sample_weight_list.append(this_weight)

sample_weights = torch.DoubleTensor(sample_weight_list)

oversampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)


# oversampled_train_loader = DataLoader(
#     imbalanced_train_set,
#     batch_size=64,
#     sampler=oversampler,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True,
# )
oversampled_train_loader = DataLoader(
    imbalanced_train_set,
    batch_size=64,
    sampler=oversampler,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

criterion_oversampling = nn.CrossEntropyLoss()

oversampling_results, oversampling_per_class_df = run_experiment(
    experiment_name="imbalanced oversampling",
    train_loader=oversampled_train_loader,
    criterion=criterion_oversampling,
    num_epochs=8,
    lr=1e-4,
)

for row in oversampling_results:
    imbalance_results.append(row)

per_class_results["imbalanced oversampling"] = oversampling_per_class_df


[ imbalanced oversampling ] Epoch 1 / 8
  Train Acc: 61.71 %
  Test Acc: 68.68 %
  Macro F1: 0.668
  Weighted F1: 0.6696
  Time: 31.8 seconds
[ imbalanced oversampling ] Epoch 2 / 8
  Train Acc: 93.61 %
  Test Acc: 74.46 %
  Macro F1: 0.7301
  Weighted F1: 0.7315
  Time: 32.0 seconds
[ imbalanced oversampling ] Epoch 3 / 8
  Train Acc: 96.12 %
  Test Acc: 74.03 %
  Macro F1: 0.7191
  Weighted F1: 0.7206
  Time: 34.1 seconds
[ imbalanced oversampling ] Epoch 4 / 8
  Train Acc: 98.23 %
  Test Acc: 74.33 %
  Macro F1: 0.7225
  Weighted F1: 0.7239
  Time: 32.0 seconds
[ imbalanced oversampling ] Epoch 5 / 8
  Train Acc: 98.98 %
  Test Acc: 74.0 %
  Macro F1: 0.7176
  Weighted F1: 0.7194
  Time: 32.3 seconds
[ imbalanced oversampling ] Epoch 6 / 8
  Train Acc: 99.73 %
  Test Acc: 75.06 %
  Macro F1: 0.7298
  Weighted F1: 0.7315
  Time: 31.7 seconds
[ imbalanced oversampling ] Epoch 7 / 8
  Train Acc: 99.73 %
  Test Acc: 74.11 %
  Macro F1: 0.7167
  Weighted F1: 0.7181
  Time: 32.0 seconds
[

## 6. Save and summarize results


In [ ]:
imbalance_df = pd.DataFrame(imbalance_results)


results_csv_path = os.path.join(PROJECT_PATH, "project1_imbalanced_results.csv")
imbalance_df.to_csv(results_csv_path, index=False)
print("Saved:", results_csv_path)


experiment_names = []
for name in imbalance_df["experiment"]:
    if name not in experiment_names:
        experiment_names.append(name)

summary_rows = []
for name in experiment_names:
    rows_for_experiment = imbalance_df[imbalance_df["experiment"] == name]

    best_index = rows_for_experiment["macro_f1"].idxmax()
    best_row = imbalance_df.loc[best_index]
    summary_rows.append(best_row)

epoch_summary_df = pd.DataFrame(summary_rows)
epoch_summary_df = epoch_summary_df.sort_values("experiment")
epoch_summary_df = epoch_summary_df.reset_index(drop=True)

summary_csv_path = os.path.join(PROJECT_PATH, "project1_imbalanced_summary.csv")
epoch_summary_df.to_csv(summary_csv_path, index=False)
print("Saved:", summary_csv_path)

display(epoch_summary_df)

# all resuls here
all_per_class_rows = []
for experiment_name in per_class_results:
    df = per_class_results[experiment_name]

    for row_number in range(len(df)):
        row = df.iloc[row_number].to_dict()
        row["experiment"] = experiment_name

        class_id = int(row["class_id"])
        if class_id in cat_class_ids:
            row["animal_type"] = "cat"
        else:
            row["animal_type"] = "dog"

        all_per_class_rows.append(row)

all_per_class_df = pd.DataFrame(all_per_class_rows)
per_class_csv_path = os.path.join(PROJECT_PATH, "project1_imbalanced_per_class_results.csv")
all_per_class_df.to_csv(per_class_csv_path, index=False)
print("Saved:", per_class_csv_path)

# cat/dog class summary
cat_dog_rows = []
for experiment_name in experiment_names:
    for animal_type in ["cat", "dog"]:
        small_df = all_per_class_df[
            (all_per_class_df["experiment"] == experiment_name)
            & (all_per_class_df["animal_type"] == animal_type)
        ]

        cat_dog_rows.append({
            "experiment": experiment_name,
            "animal_type": animal_type,
            "mean_per_class_accuracy": small_df["per_class_accuracy"].mean(),
            "mean_f1": small_df["f1"].mean(),
            "mean_recall": small_df["recall"].mean(),
            "mean_precision": small_df["precision"].mean(),
        })

cat_dog_summary = pd.DataFrame(cat_dog_rows)
cat_dog_csv_path = os.path.join(PROJECT_PATH, "project1_imbalanced_cat_dog_summary.csv")
cat_dog_summary.to_csv(cat_dog_csv_path, index=False)
print("Saved:", cat_dog_csv_path)

display(cat_dog_summary)


Saved: /content/drive/MyDrive/deep learning/DD2424_Project1/project1_imbalanced_results.csv
Saved: /content/drive/MyDrive/deep learning/DD2424_Project1/project1_imbalanced_summary.csv


,experiment,epoch,train_loss,train_acc,test_loss,test_acc,macro_f1,weighted_f1,time_sec
0,imbalanced baseline,8,0.035583,100.000000,0.965705,72.444808,0.700834,0.703137,31.447814
1,imbalanced oversampling,2,0.554378,93.605335,1.158312,74.461706,0.730116,0.731492,31.995806
2,imbalanced weighted cross entropy,5,0.207762,97.803060,1.079221,75.579177,0.741377,0.742647,31.354117


Saved: /content/drive/MyDrive/deep learning/DD2424_Project1/project1_imbalanced_per_class_results.csv
Saved: /content/drive/MyDrive/deep learning/DD2424_Project1/project1_imbalanced_cat_dog_summary.csv


,experiment,animal_type,mean_per_class_accuracy,mean_f1,mean_recall,mean_precision
0,imbalanced baseline,cat,35.418241,0.443920,0.354182,0.713624
1,imbalanced baseline,dog,89.827062,0.824153,0.898271,0.778100
2,imbalanced weighted cross entropy,cat,46.686179,0.550640,0.466862,0.725184
3,imbalanced weighted cross entropy,dog,89.161938,0.832931,0.891619,0.791474
4,imbalanced oversampling,cat,46.509178,0.536580,0.465092,0.686529
5,imbalanced oversampling,dog,87.562483,0.823013,0.875625,0.786851
